# Miniprojeto — Treinando um sistema generativo de imagem a partir de texto
**Objetivo:** ensinar um modelo de difusão a gerar **um produto ou estilo específico** a partir de poucas fotos,
usando **DreamBooth + LoRA**.

**Formato:** duplas · entrega em 2 semanas · apresentação de 5 minutos na aula seguinte.

### A ideia
O Stable Diffusion sabe desenhar "uma garrafa térmica", mas não **a** garrafa térmica da sua empresa.
O DreamBooth associa um identificador raro (ex.: `sks`) ao seu objeto. A LoRA treina só ~0,1% dos pesos,
o que cabe numa T4 gratuita em 15 a 25 minutos.

### O que entregar
1. Este notebook executado, com as imagens **antes × depois** com os mesmos prompts e sementes.
2. Uma página com o caso de negócio: qual problema a geração resolve, custo estimado e riscos.
3. Uma apresentação de 5 minutos.

### Rubrica (10 pontos)
| Critério | Pontos | O que se espera |
|---|---|---|
| Dados | 2 | 5–20 fotos próprias ou com direito de uso, variadas (ângulos, fundos, luz), com a curadoria explicada |
| Treino | 2 | hiperparâmetros documentados; pelo menos **2 experimentos** comparados (ex.: passos, rank, learning rate) |
| Avaliação | 3 | antes × depois com prompts e sementes fixos; fidelidade ao objeto × diversidade de cenários; sinais de *overfitting* |
| Negócio e ética | 2 | caso de uso, custo, licença do modelo base, direitos de imagem, risco de uso indevido e viés |
| Apresentação | 1 | clareza e cumprimento do tempo |

> ⚠️ Precisa de **GPU** (Colab → Ambiente de execução → Alterar tipo → T4).
> Não use fotos de pessoas sem consentimento por escrito.

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
MODO_RAPIDO = True   # True = subamostra para caber nos tempos da aula
SMOKE = os.environ.get("AULA_SMOKE") == "1"   # só para teste automatizado em CPU

if IN_COLAB:
    !pip install -q -U diffusers transformers accelerate peft bitsandbytes ftfy tensorboard

import torch
import transformers
transformers.logging.set_verbosity_error()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| MODO_RAPIDO:", MODO_RAPIDO)
if DEVICE == "cpu" and not SMOKE:
    print("⚠️  Sem GPU. No Colab: Ambiente de execução → Alterar tipo → GPU T4.")

## 1. Configuração do projeto
Escolha o identificador raro (`sks`) e a **classe** do objeto (`dog`, `bottle`, `sneaker`, `backpack`...).
Os prompts ficam em inglês, porque o text encoder (CLIP) entende inglês melhor.

In [ ]:
from pathlib import Path

# TODO: defina a classe do seu objeto e o prompt de instância
...
MODELO_BASE = "stable-diffusion-v1-5/stable-diffusion-v1-5"   # licença CreativeML OpenRAIL-M
PASTA_FOTOS = Path("fotos_instancia")
PASTA_LORA = Path("lora_dreambooth")
PASTA_FOTOS.mkdir(exist_ok=True)

PROMPTS_AVALIACAO = [
    f"a photo of sks {CLASSE} on the beach",
    f"a photo of sks {CLASSE} in a modern kitchen, product photography",
    f"a watercolor painting of sks {CLASSE}",
    f"sks {CLASSE} on a billboard in Times Square",
]
SEMENTES = [0, 1]

## 2. As fotos
**Opção A (o projeto de verdade):** envie de 5 a 20 fotos suas do objeto. Varie ângulo, fundo e luz,
com o objeto centralizado e nítido.
**Opção B (para testar o pipeline):** use as 5 fotos de cachorro de exemplo da Hugging Face.

In [ ]:
from PIL import Image

USAR_EXEMPLO = True   # mude para False para enviar suas fotos

if USAR_EXEMPLO:
    from huggingface_hub import snapshot_download
    snapshot_download("diffusers/dog-example", repo_type="dataset", local_dir=PASTA_FOTOS,
                      ignore_patterns=[".gitattributes", "*.md"])
elif IN_COLAB:
    from google.colab import files
    for nome, conteudo in files.upload().items():
        (PASTA_FOTOS / nome).write_bytes(conteudo)

fotos = sorted(p for p in PASTA_FOTOS.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print(len(fotos), "fotos")

import matplotlib.pyplot as plt
fig, eixos = plt.subplots(1, len(fotos), figsize=(3 * len(fotos), 3))
for eixo, p in zip(eixos, fotos):
    eixo.imshow(Image.open(p)); eixo.axis("off")

## 3. Antes: o que o modelo base gera com o seu prompt?
Sem treino, `sks` não significa nada. O modelo gera um objeto genérico da classe.

In [ ]:
from diffusers import StableDiffusionPipeline

def gerar_grade(pipe, titulo):
    imagens = [[pipe(p, num_inference_steps=30, guidance_scale=7.5,
                     generator=torch.Generator("cuda").manual_seed(s)).images[0] for s in SEMENTES]
               for p in PROMPTS_AVALIACAO]
    fig, eixos = plt.subplots(len(SEMENTES), len(PROMPTS_AVALIACAO), figsize=(4 * len(PROMPTS_AVALIACAO), 4 * len(SEMENTES)))
    for j, p in enumerate(PROMPTS_AVALIACAO):
        for i, s in enumerate(SEMENTES):
            eixos[i, j].imshow(imagens[j][i]); eixos[i, j].axis("off")
            if i == 0:
                eixos[i, j].set_title(p, fontsize=9, wrap=True)
    plt.suptitle(titulo); plt.tight_layout(); plt.show()

pipe = StableDiffusionPipeline.from_pretrained(MODELO_BASE, torch_dtype=torch.float16, safety_checker=None).to("cuda")
gerar_grade(pipe, "ANTES do treino")
del pipe; torch.cuda.empty_cache()

## 4. Treino DreamBooth + LoRA
Usamos o script oficial de exemplo do `diffusers`, na versão que bate com a biblioteca instalada.

| Hiperparâmetro | Valor inicial | Efeito |
|---|---|---|
| `max_train_steps` | 400 | mais passos dão mais fidelidade, mas arriscam *overfitting* (o objeto aparece igual em tudo) |
| `learning_rate` | 1e-4 | alto demais "queima" o modelo |
| `rank` | 8 | capacidade da LoRA |

⏱️ ~15 a 25 minutos numa T4.

In [ ]:
import diffusers

versao = diffusers.__version__.split(".dev")[0]
!wget -q -O train_dreambooth_lora.py https://raw.githubusercontent.com/huggingface/diffusers/v{versao}/examples/dreambooth/train_dreambooth_lora.py
!accelerate config default

# TODO: ajuste os hiperparâmetros dos seus experimentos
...

In [ ]:
!accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path={MODELO_BASE} \
  --instance_data_dir={PASTA_FOTOS} \
  --instance_prompt="{INSTANCIA}" \
  --output_dir={PASTA_LORA} \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --gradient_checkpointing \
  --use_8bit_adam \
  --mixed_precision=fp16 \
  --learning_rate={LR} \
  --lr_scheduler=constant \
  --lr_warmup_steps=0 \
  --rank={RANK} \
  --max_train_steps={PASSOS} \
  --checkpointing_steps=100000 \
  --seed=42

## 5. Depois: as mesmas sementes e prompts, agora com a LoRA

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(MODELO_BASE, torch_dtype=torch.float16, safety_checker=None).to("cuda")
pipe.load_lora_weights(PASTA_LORA)
gerar_grade(pipe, f"DEPOIS: LoRA com {PASSOS} passos, lr={LR}, rank={RANK}")

## 6. Análise (vale 3 pontos)
Responda nesta célula:
1. **Fidelidade:** o objeto gerado é reconhecível como o seu? Em quais prompts falha?
2. **Diversidade:** o modelo ainda obedece ao cenário (praia, cozinha, aquarela) ou só repete as fotos de treino?
3. **Experimentos:** o que mudou entre as suas duas configurações de hiperparâmetros?
4. **Negócio:** quanto custaria gerar 1.000 imagens de catálogo assim, comparado a uma sessão de fotos?
   Quais riscos (licença do modelo base, direitos de imagem, deepfakes) a empresa precisa endereçar?